# Zomato Analytics — Synthetic Data Generator

**Purpose**: Generate realistic synthetic data using Faker and write Parquet files
to the Unity Catalog Volume for Bronze layer ingestion.

**Output**: Parquet files at `/Volumes/{catalog}/raw/landing/{entity}/`

**Entities Generated**:
- customers (10,000)
- restaurants (2,500)
- orders (150,000)
- deliveries (~110,000)
- reviews (120,000)
- delivery partners (5,001) → **Lakebase Postgres** table + app changes on every run (Exercise 5)

## Exercise 5 — the Lakebase source: what this notebook does

**The use case.** The delivery-partner master (name, city, vehicle, status, rating, KYC …) lives in a **Lakebase
Postgres** table that the partner app changes all day. We copy it into the lakehouse and keep the history of every
partner. Instead of copying the whole table on every run, we copy **only the rows that changed since the last run**.
To know what *"since the last run"* means we keep a **watermark**: the newest `updated_at` value already loaded.

```
LAKEBASE (Postgres)   zomato_ops.delivery_partners      one row per partner   ◀── THIS NOTEBOOK plays the partner app
                      updated_at = time of the row's last change  (the watermark column)
      │
      │  ① BRONZE   full        → read ALL partners
      │             incremental → read only partners with updated_at > watermark
      ▼
bronze.delivery_partners              every partner row received from Lakebase
bronze.delivery_partners_watermark    ONE row: the newest updated_at loaded so far
      │
      │  ② SILVER   newest row per partner, cleaned → overwrite
      ▼
silver.delivery_partners              one row per partner = how the partner looks NOW
      │
      │  ③ GOLD     compare with the current version → SCD Type 2
      ▼
gold.dim_delivery_partners            one row per partner per version = full history
```

**Section 8 plays the partner app.** Only one table is created in Lakebase:

1. Create the Lakebase project if it does not exist yet (first run only).
2. Create the table `zomato_ops.delivery_partners`. Its `updated_at` column holds the time of the row's last change —
   this is the **watermark column** that Bronze uses.
3. First run only: add 5,001 partners (same IDs as `delivery_partner_id` in the deliveries data).
4. Every run: make app changes — new partners join, partners change city / vehicle / status / rating, some leave.
   Every change sets `updated_at = now()`. Leaving is a **soft delete** (`partner_status = 'OFFBOARDED'`, the row
   stays), so Bronze sees it as a changed row too.

### 🔐 Connecting securely — where the credentials live

| # | Rule | How it is done here |
|---|---|---|
| 1 | **No passwords or tokens in code, widgets, job parameters or Git** | Nothing secret appears in any notebook |
| 2 | **Use the identity that runs the notebook / job** | `WorkspaceClient()` picks it up automatically — you when you run it by hand, the job's *Run as* identity when scheduled (use a **service principal** for jobs) |
| 3 | **Short-lived database login** | `w.postgres.generate_database_credential(...)` returns an **OAuth token valid for 1 hour**; it is used as the Postgres password and never printed or saved |
| 4 | **Encrypted connection** | `sslmode="require"` |
| 5 | **Least privilege** | The pipeline only needs `SELECT` on the partner table (SQL below) |
| 6 | **Static password only if OAuth is impossible** | Keep it in a **Databricks secret scope** and read it with `dbutils.secrets.get(...)` — it is shown as `[REDACTED]` |
| 7 | **Your laptop / CLI** | `databricks auth login --host <workspace-url>` (browser OAuth) |
| 8 | **CI/CD** | GitHub Actions → Databricks with **OIDC workload identity federation**, not a personal access token |

**Once, when the scheduled job runs as a service principal (SP)**
1. Give the SP permission on the Lakebase project (project → *Permissions*).
2. Add a Postgres OAuth role for the SP on the branch (Lakebase UI → *Roles*).
3. As the project owner, grant only what it needs:
   ```sql
   GRANT USAGE  ON SCHEMA zomato_ops TO "<sp-application-id>";
   GRANT SELECT ON zomato_ops.delivery_partners TO "<sp-application-id>";
   -- only the identity that runs this setup notebook (it plays the app) also needs:
   -- GRANT INSERT, UPDATE ON zomato_ops.delivery_partners TO "<sp-application-id>";
   ```

**Static-password fallback (only when OAuth cannot be used)**
```bash
databricks secrets create-scope zomato-lakebase
databricks secrets put-secret zomato-lakebase pg-password    # prompts for the value - never type it on the command line
```
```python
pg_password = dbutils.secrets.get(scope="zomato-lakebase", key="pg-password")    # appears as [REDACTED]
```

## 1. Install Dependencies

In [ ]:
%pip install faker "psycopg[binary]>=3.1" "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

## 2. Configuration

In [ ]:
import hashlib
import random
from datetime import datetime, timedelta, timezone

from faker import Faker
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType,
    IntegerType, FloatType, ArrayType,
)

In [ ]:
dbutils.widgets.text("catalog_name", "zomato_analytics", "Unity Catalog name")
dbutils.widgets.text("num_customers", "10000", "Number of customers")
dbutils.widgets.text("num_restaurants", "2500", "Number of restaurants")
dbutils.widgets.text("num_orders", "150000", "Number of orders")
dbutils.widgets.text("num_reviews", "120000", "Number of reviews")

CATALOG = dbutils.widgets.get("catalog_name")
NUM_CUSTOMERS = int(dbutils.widgets.get("num_customers"))
NUM_RESTAURANTS = int(dbutils.widgets.get("num_restaurants"))
NUM_ORDERS = int(dbutils.widgets.get("num_orders"))
NUM_REVIEWS = int(dbutils.widgets.get("num_reviews"))

LANDING_PATH = f"/Volumes/{CATALOG}/raw/landing"

print(f"Catalog      : {CATALOG}")
print(f"Landing Path : {LANDING_PATH}")
print(f"Customers    : {NUM_CUSTOMERS:,}")
print(f"Restaurants  : {NUM_RESTAURANTS:,}")
print(f"Orders       : {NUM_ORDERS:,}")
print(f"Reviews      : {NUM_REVIEWS:,}")

## 3. Domain Configuration

In [ ]:
CITIES = [
    "Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai",
    "Kolkata", "Pune", "Ahmedabad", "Jaipur", "Lucknow",
    "Chandigarh", "Goa", "Kochi", "Indore", "Nagpur",
]

CUISINES = [
    "North Indian", "South Indian", "Chinese", "Italian", "Continental",
    "Mughlai", "Street Food", "Biryani", "Desserts", "Beverages",
    "Fast Food", "Thai", "Japanese", "Mexican", "Mediterranean",
    "Korean", "Bengali", "Rajasthani", "Gujarati", "Kerala",
]

RESTAURANT_TYPES = [
    "Quick Bites", "Casual Dining", "Fine Dining", "Cafe", "Bakery",
    "Cloud Kitchen", "Food Court", "Dhaba", "Lounge", "Microbrewery",
]

PAYMENT_METHODS = ["UPI", "Credit Card", "Debit Card", "Cash on Delivery", "Wallet", "Net Banking"]
ORDER_STATUSES = ["Placed", "Confirmed", "Preparing", "Out for Delivery", "Delivered", "Cancelled", "Refunded"]
DELIVERY_STATUSES = ["Assigned", "Picked Up", "In Transit", "Delivered", "Failed", "Returned"]
PLATFORMS = ["Android App", "iOS App", "Web", "Partner API"]
SUBSCRIPTION_TIERS = ["Free", "Zomato Pro", "Zomato Pro Plus", "Zomato Gold"]

DATA_START = "2024-01-01"
DATA_END = "2025-12-31"

## 4. Generator Functions

In [ ]:
fake = Faker("en_IN")
Faker.seed(42)
random.seed(42)
UTC = timezone.utc


def _gen_id(prefix, index):
    raw = f"{prefix}_{index}"
    return f"{prefix.upper()}_{hashlib.md5(raw.encode()).hexdigest()[:8]}"


def _rand_ts(start, end):
    s = datetime.fromisoformat(start)
    e = datetime.fromisoformat(end)
    return s + timedelta(seconds=random.random() * (e - s).total_seconds())

In [ ]:
def generate_customers():
    print("  Generating customers...")
    rows = []
    for i in range(NUM_CUSTOMERS):
        signup = _rand_ts(DATA_START, DATA_END)
        rows.append({
            "customer_id": _gen_id("cust", i),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "city": random.choice(CITIES),
            "locality": fake.street_name(),
            "pincode": fake.postcode(),
            "latitude": round(fake.latitude(), 6),
            "longitude": round(fake.longitude(), 6),
            "signup_date": signup.isoformat(),
            "subscription_tier": random.choice(SUBSCRIPTION_TIERS),
            "is_active": random.choices([True, False], weights=[0.85, 0.15])[0],
            "preferred_payment": random.choice(PAYMENT_METHODS),
            "preferred_cuisine": random.choice(CUISINES),
            "total_orders_lifetime": random.randint(0, 350),
            "avg_order_value": round(random.uniform(150, 900), 2),
            "_ingested_at": datetime.now(UTC).isoformat(),
            "_source_system": "zomato_app_backend",
        })
    return rows

In [ ]:
def generate_restaurants():
    print("  Generating restaurants...")
    rows = []
    for i in range(NUM_RESTAURANTS):
        cuisines = random.sample(CUISINES, random.randint(1, 4))
        onboarded = _rand_ts("2020-01-01", DATA_END)
        rows.append({
            "restaurant_id": _gen_id("rest", i),
            "name": f"{fake.company()} {random.choice(['Kitchen', 'Bistro', 'Dhaba', 'Cafe', 'Restaurant', 'Eatery', 'House'])}",
            "city": random.choice(CITIES),
            "locality": fake.street_name(),
            "address": fake.address().replace("\n", ", "),
            "pincode": fake.postcode(),
            "latitude": round(fake.latitude(), 6),
            "longitude": round(fake.longitude(), 6),
            "cuisines": cuisines,
            "restaurant_type": random.choice(RESTAURANT_TYPES),
            "avg_cost_for_two": random.choice([200, 300, 500, 700, 1000, 1500, 2000, 2500]),
            "rating": round(random.uniform(2.5, 5.0), 1),
            "total_reviews": random.randint(10, 5000),
            "is_delivering_now": random.choices([True, False], weights=[0.8, 0.2])[0],
            "has_online_delivery": random.choices([True, False], weights=[0.9, 0.1])[0],
            "has_table_booking": random.choices([True, False], weights=[0.3, 0.7])[0],
            "is_premium_partner": random.choices([True, False], weights=[0.15, 0.85])[0],
            "onboarded_date": onboarded.isoformat(),
            "owner_name": fake.name(),
            "owner_phone": fake.phone_number(),
            "gst_number": f"{''.join(random.choices('0123456789', k=2))}{fake.bothify('?????').upper()}{''.join(random.choices('0123456789', k=4))}{fake.bothify('?').upper()}{''.join(random.choices('0123456789AZ', k=2))}",
            "_ingested_at": datetime.now(UTC).isoformat(),
            "_source_system": "restaurant_onboarding_svc",
        })
    return rows

In [ ]:
def generate_orders(customers, restaurants):
    print("  Generating orders...")
    orders = []
    for i in range(NUM_ORDERS):
        customer = random.choice(customers)
        restaurant = random.choice(restaurants)
        order_ts = _rand_ts(DATA_START, DATA_END)
        status = random.choices(ORDER_STATUSES, weights=[0.05, 0.05, 0.05, 0.05, 0.70, 0.07, 0.03])[0]

        num_items = random.randint(1, 5)
        subtotal = round(sum(random.uniform(49, 1499) for _ in range(num_items)), 2)

        discount_pct = 0.0
        coupon_code = None
        if random.random() < 0.35:
            discount_pct = round(random.uniform(5, 50), 1)
            coupon_code = fake.bothify("ZOMATO##??").upper()

        discount_amount = round(subtotal * discount_pct / 100, 2)
        delivery_fee = round(random.uniform(0, 80), 2)
        tax_amount = round((subtotal - discount_amount) * 0.05, 2)
        total_amount = round(subtotal - discount_amount + delivery_fee + tax_amount, 2)

        orders.append({
            "order_id": _gen_id("ord", i),
            "customer_id": customer["customer_id"],
            "restaurant_id": restaurant["restaurant_id"],
            "order_placed_at": order_ts.isoformat(),
            "order_status": status,
            "payment_method": random.choice(PAYMENT_METHODS),
            "payment_status": "Completed" if status == "Delivered" else random.choice(["Pending", "Completed", "Failed"]),
            "platform": random.choice(PLATFORMS),
            "subtotal": subtotal,
            "discount_pct": discount_pct,
            "discount_amount": discount_amount,
            "coupon_code": coupon_code,
            "delivery_fee": delivery_fee,
            "tax_amount": tax_amount,
            "total_amount": total_amount,
            "num_items": num_items,
            "special_instructions": fake.sentence(nb_words=5) if random.random() < 0.2 else None,
            "_ingested_at": datetime.now(UTC).isoformat(),
            "_source_system": "order_management_svc",
        })
    return orders

In [ ]:
def generate_deliveries(orders):
    print("  Generating deliveries...")
    deliverable = [o for o in orders if o["order_status"] in ("Out for Delivery", "Delivered")]
    deliveries = []
    for i, order in enumerate(deliverable):
        pickup_ts = datetime.fromisoformat(order["order_placed_at"]) + timedelta(minutes=random.randint(5, 25))
        delivery_time = random.randint(15, 60)
        delivered_ts = pickup_ts + timedelta(minutes=delivery_time)

        deliveries.append({
            "delivery_id": _gen_id("dlv", i),
            "order_id": order["order_id"],
            "delivery_partner_id": _gen_id("dp", random.randint(0, 5000)),
            "delivery_partner_name": fake.name(),
            "partner_phone": fake.phone_number(),
            "vehicle_type": random.choice(["Bike", "Scooter", "Bicycle", "Car"]),
            "pickup_timestamp": pickup_ts.isoformat(),
            "delivered_timestamp": delivered_ts.isoformat(),
            "delivery_time_mins": delivery_time,
            "delivery_status": random.choices(DELIVERY_STATUSES, weights=[0.03, 0.05, 0.05, 0.80, 0.05, 0.02])[0],
            "delivery_distance_km": round(random.uniform(0.5, 15.0), 2),
            "delivery_rating": random.choices([1, 2, 3, 4, 5, None], weights=[0.02, 0.05, 0.10, 0.35, 0.40, 0.08])[0],
            "_ingested_at": datetime.now(UTC).isoformat(),
            "_source_system": "logistics_svc",
        })
    return deliveries

In [ ]:
def generate_reviews(orders):
    print("  Generating reviews...")
    delivered = [o for o in orders if o["order_status"] == "Delivered"]
    sentiments = [
        "Amazing food, will order again!", "Decent taste but took too long.",
        "Cold food received, very disappointed.", "Best biryani in the city!",
        "Average experience, nothing special.", "Packaging was excellent, food was fresh.",
        "Portion size too small for the price.", "Loved the variety of options.",
        "Quick delivery, hot and fresh food!", "Great value for money.",
        "Consistent quality every time I order.", "Would recommend to friends and family.",
    ]
    reviews = []
    for i in range(min(NUM_REVIEWS, len(delivered))):
        order = delivered[i]
        review_ts = datetime.fromisoformat(order["order_placed_at"]) + timedelta(hours=random.randint(1, 48))
        rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.08, 0.15, 0.35, 0.37])[0]
        reviews.append({
            "review_id": _gen_id("rev", i),
            "order_id": order["order_id"],
            "customer_id": order["customer_id"],
            "restaurant_id": order["restaurant_id"],
            "rating": rating,
            "review_text": random.choice(sentiments) if random.random() < 0.7 else fake.paragraph(nb_sentences=2),
            "review_timestamp": review_ts.isoformat(),
            "sentiment_label": "Positive" if rating >= 4 else ("Negative" if rating <= 2 else "Neutral"),
            "is_verified_order": True,
            "upvotes": random.randint(0, 50),
            "has_photo": random.choices([True, False], weights=[0.2, 0.8])[0],
            "_ingested_at": datetime.now(UTC).isoformat(),
            "_source_system": "review_svc",
        })
    return reviews

## 5. Generate All Data

In [ ]:
import time

overall_start = time.time()

print("=" * 60)
print("  Zomato Analytics — Data Generation")
print("=" * 60)

customers = generate_customers()
print(f"    ✓ Customers: {len(customers):,}")

restaurants = generate_restaurants()
print(f"    ✓ Restaurants: {len(restaurants):,}")

orders = generate_orders(customers, restaurants)
print(f"    ✓ Orders: {len(orders):,}")

deliveries = generate_deliveries(orders)
print(f"    ✓ Deliveries: {len(deliveries):,}")

reviews = generate_reviews(orders)
print(f"    ✓ Reviews: {len(reviews):,}")

gen_time = round(time.time() - overall_start, 1)
print(f"\n  Data generation completed in {gen_time}s")

## 6. Write Parquet to Landing Volume

In [ ]:
# Convert Decimal values to float for customers and restaurants
for cust in customers:
    cust['latitude'] = float(cust['latitude'])
    cust['longitude'] = float(cust['longitude'])
    cust['avg_order_value'] = float(cust['avg_order_value'])

for rest in restaurants:
    rest['latitude'] = float(rest['latitude'])
    rest['longitude'] = float(rest['longitude'])

In [ ]:
def write_to_landing(data, entity_name, schema):
    """Convert list of dicts to Spark DataFrame and write as Parquet to Volume."""
    path = f"{LANDING_PATH}/{entity_name}"
    df = spark.createDataFrame(data, schema=schema)
    df.write.mode("overwrite").parquet(path)
    count = df.count()
    print(f"  ✓ {entity_name}: {count:,} records → {path}")
    return count

In [ ]:
from pyspark.sql.types import LongType

LANDING_SCHEMAS = {
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("first_name", StringType()), StructField("last_name", StringType()),
        StructField("email", StringType()), StructField("phone", StringType()),
        StructField("city", StringType()), StructField("locality", StringType()),
        StructField("pincode", StringType()),
        StructField("latitude", DoubleType()), StructField("longitude", DoubleType()),
        StructField("signup_date", StringType()),
        StructField("subscription_tier", StringType()),
        StructField("is_active", BooleanType()),
        StructField("preferred_payment", StringType()),
        StructField("preferred_cuisine", StringType()),
        StructField("total_orders_lifetime", IntegerType()),
        StructField("avg_order_value", DoubleType()),
        StructField("_ingested_at", StringType()),
        StructField("_source_system", StringType()),
    ]),
    "restaurants": StructType([
        StructField("restaurant_id", StringType(), False),
        StructField("name", StringType()), StructField("city", StringType()),
        StructField("locality", StringType()), StructField("address", StringType()),
        StructField("pincode", StringType()),
        StructField("latitude", DoubleType()), StructField("longitude", DoubleType()),
        StructField("cuisines", ArrayType(StringType())),
        StructField("restaurant_type", StringType()),
        StructField("avg_cost_for_two", IntegerType()),
        StructField("rating", FloatType()),
        StructField("total_reviews", IntegerType()),
        StructField("is_delivering_now", BooleanType()),
        StructField("has_online_delivery", BooleanType()),
        StructField("has_table_booking", BooleanType()),
        StructField("is_premium_partner", BooleanType()),
        StructField("onboarded_date", StringType()),
        StructField("owner_name", StringType()), StructField("owner_phone", StringType()),
        StructField("gst_number", StringType()),
        StructField("_ingested_at", StringType()),
        StructField("_source_system", StringType()),
    ]),
    "orders": StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType()), StructField("restaurant_id", StringType()),
        StructField("order_placed_at", StringType()),
        StructField("order_status", StringType()),
        StructField("payment_method", StringType()), StructField("payment_status", StringType()),
        StructField("platform", StringType()),
        StructField("subtotal", DoubleType()), StructField("discount_pct", DoubleType()),
        StructField("discount_amount", DoubleType()), StructField("coupon_code", StringType()),
        StructField("delivery_fee", DoubleType()), StructField("tax_amount", DoubleType()),
        StructField("total_amount", DoubleType()),
        StructField("num_items", IntegerType()),
        StructField("special_instructions", StringType()),
        StructField("_ingested_at", StringType()),
        StructField("_source_system", StringType()),
    ]),
    "deliveries": StructType([
        StructField("delivery_id", StringType(), False),
        StructField("order_id", StringType()),
        StructField("delivery_partner_id", StringType()),
        StructField("delivery_partner_name", StringType()),
        StructField("partner_phone", StringType()),
        StructField("vehicle_type", StringType()),
        StructField("pickup_timestamp", StringType()),
        StructField("delivered_timestamp", StringType()),
        StructField("delivery_time_mins", IntegerType()),
        StructField("delivery_status", StringType()),
        StructField("delivery_distance_km", DoubleType()),
        StructField("delivery_rating", IntegerType()),
        StructField("_ingested_at", StringType()),
        StructField("_source_system", StringType()),
    ]),
    "reviews": StructType([
        StructField("review_id", StringType(), False),
        StructField("order_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("restaurant_id", StringType()),
        StructField("rating", IntegerType()),
        StructField("review_text", StringType()),
        StructField("review_timestamp", StringType()),
        StructField("sentiment_label", StringType()),
        StructField("is_verified_order", BooleanType()),
        StructField("upvotes", IntegerType()),
        StructField("has_photo", BooleanType()),
        StructField("_ingested_at", StringType()),
        StructField("_source_system", StringType()),
    ]),
}

In [ ]:
print("\n" + "=" * 60)
print("  Writing Parquet to Landing Volume")
print("=" * 60)

total_written = 0
total_written += write_to_landing(customers, "customers", LANDING_SCHEMAS["customers"])
total_written += write_to_landing(restaurants, "restaurants", LANDING_SCHEMAS["restaurants"])
total_written += write_to_landing(orders, "orders", LANDING_SCHEMAS["orders"])
total_written += write_to_landing(deliveries, "deliveries", LANDING_SCHEMAS["deliveries"])
total_written += write_to_landing(reviews, "reviews", LANDING_SCHEMAS["reviews"])

total_time = round(time.time() - overall_start, 1)

print(f"\n{'='*60}")
print(f"  ✓ Total records written : {total_written:,}")
print(f"  ✓ Landing path          : {LANDING_PATH}")
print(f"  ✓ Total time            : {total_time}s")
print(f"{'='*60}")

## 7. Verify Landing Data

In [ ]:
print("Verifying landing data...")
for entity in ["customers", "restaurants", "orders", "deliveries", "reviews"]:
    path = f"{LANDING_PATH}/{entity}"
    df = spark.read.parquet(path)
    print(f"  ✓ {entity}: {df.count():,} records")

print("\n🟢 Data generation and landing complete.")

## 8. Lakebase Source — Delivery Partners (Exercise 5)

Steps 1 → 4 from the overview at the top of this notebook.

In [ ]:
dbutils.widgets.text("lakebase_project_id", "zomato-ops", "Lakebase project id")
dbutils.widgets.text("lakebase_branch_id", "production", "Lakebase branch id")
dbutils.widgets.text("lakebase_database", "databricks_postgres", "Lakebase Postgres database")
dbutils.widgets.text("num_partner_changes", "100", "Partner changes to simulate per run")

LAKEBASE_PROJECT_ID = dbutils.widgets.get("lakebase_project_id")
LAKEBASE_BRANCH_ID = dbutils.widgets.get("lakebase_branch_id")
LAKEBASE_DATABASE = dbutils.widgets.get("lakebase_database")
NUM_PARTNER_CHANGES = int(dbutils.widgets.get("num_partner_changes"))
NUM_DELIVERY_PARTNERS = 5001  # deliveries use partner ids _gen_id("dp", 0 … 5000)

print(f"Lakebase        : projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH_ID}, database {LAKEBASE_DATABASE}")
print(f"Changes per run : {NUM_PARTNER_CHANGES}")

### 8.1 Secure connection helper

In [ ]:
import time

import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import EndpointType

# WorkspaceClient() automatically uses the identity running this notebook / job.
# No token or password is written anywhere in this notebook.
w = WorkspaceClient()


def find_read_write_endpoint():
    """Return the read-write compute endpoint of our Lakebase branch."""
    branch_path = f"projects/{LAKEBASE_PROJECT_ID}/branches/{LAKEBASE_BRANCH_ID}"
    for endpoint in w.postgres.list_endpoints(parent=branch_path):
        if endpoint.status.endpoint_type == EndpointType.ENDPOINT_TYPE_READ_WRITE:
            return endpoint
    raise RuntimeError(f"No read-write endpoint found on {branch_path}")


def get_lakebase_connection():
    """
    Open a secure connection to Lakebase:
      1. find the database host (the read-write endpoint)
      2. ask Databricks for a short-lived OAuth token (valid 1 hour) and use it as the password
      3. connect with SSL; retry a few times because an idle database may need a moment to wake up
    """
    endpoint = find_read_write_endpoint()
    user_name = w.current_user.me().user_name

    for attempt in range(1, 4):
        try:
            credential = w.postgres.generate_database_credential(endpoint=endpoint.name)
            return psycopg.connect(
                host=endpoint.status.hosts.host,
                port=5432,
                dbname=LAKEBASE_DATABASE,
                user=user_name,
                password=credential.token,  # short-lived OAuth token, never printed or stored
                sslmode="require",          # encrypted connection only
                autocommit=True,
            )
        except psycopg.OperationalError as error:
            print(f"  … Lakebase not ready yet (attempt {attempt}/3): {error}")
            time.sleep(attempt * 5)
    raise RuntimeError("Could not connect to Lakebase after 3 attempts")

### 8.2 Create the Lakebase project and the partner table (safe to re-run)

In [ ]:
from databricks.sdk.errors import NotFound
from databricks.sdk.service.postgres import Project, ProjectSpec


def create_lakebase_project_if_missing():
    """Reuse the Lakebase project if it exists; otherwise create it (first run only, takes a few minutes)."""
    try:
        w.postgres.get_project(name=f"projects/{LAKEBASE_PROJECT_ID}")
        print(f"  ✓ Using existing Lakebase project '{LAKEBASE_PROJECT_ID}'")
    except NotFound:
        print(f"  … Creating Lakebase project '{LAKEBASE_PROJECT_ID}' (Postgres 17) — first run only")
        new_project = Project(spec=ProjectSpec(display_name="Zomato Ops (OLTP)", pg_version=17))
        w.postgres.create_project(project=new_project, project_id=LAKEBASE_PROJECT_ID).wait()
        print(f"  ✓ Created Lakebase project '{LAKEBASE_PROJECT_ID}'")


CREATE_PARTNER_TABLE_SQL = """
CREATE SCHEMA IF NOT EXISTS zomato_ops;

CREATE TABLE IF NOT EXISTS zomato_ops.delivery_partners (
    partner_id        TEXT PRIMARY KEY,
    partner_name      TEXT NOT NULL,
    phone             TEXT,
    email             TEXT,
    city              TEXT NOT NULL,
    vehicle_type      TEXT NOT NULL,
    partner_status    TEXT NOT NULL,          -- ONBOARDING / ACTIVE / INACTIVE / SUSPENDED / OFFBOARDED
    shift_preference  TEXT,
    rating            NUMERIC(3,2),
    joined_date       DATE NOT NULL,
    kyc_verified      BOOLEAN NOT NULL DEFAULT FALSE,
    created_at        TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at        TIMESTAMPTZ NOT NULL DEFAULT now()   -- the watermark column: set on every change
);

-- makes "WHERE updated_at > watermark" fast
CREATE INDEX IF NOT EXISTS ix_delivery_partners_updated_at ON zomato_ops.delivery_partners (updated_at);
"""

create_lakebase_project_if_missing()
with get_lakebase_connection() as conn:
    conn.execute(CREATE_PARTNER_TABLE_SQL)
print("  ✓ Lakebase table zomato_ops.delivery_partners is ready")

### 8.3 Add partners (first run) and simulate app changes (every run)

In [ ]:
from datetime import date

VEHICLE_TYPES = ["Bike", "Scooter", "Bicycle", "Car"]  # same values as the deliveries data
SHIFTS = ["Morning", "Afternoon", "Evening", "Night", "Flexible"]

INSERT_PARTNER_SQL = """
    INSERT INTO zomato_ops.delivery_partners
        (partner_id, partner_name, phone, email, city, vehicle_type,
         partner_status, shift_preference, rating, joined_date, kyc_verified)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (partner_id) DO NOTHING
"""

# Leaving the platform is a soft delete: the row stays, only the status (and updated_at) change
OFFBOARD_PARTNER_SQL = """
    UPDATE zomato_ops.delivery_partners
    SET partner_status = 'OFFBOARDED', updated_at = now()
    WHERE partner_id = %s
"""


def make_partner(index, rng, faker, is_new_joiner=False):
    """Build one partner row (same column order as INSERT_PARTNER_SQL)."""
    partner_id = _gen_id("dp", index)  # same ID format as deliveries.delivery_partner_id
    if is_new_joiner:
        status, rating, joined, kyc = "ONBOARDING", None, date.today(), False
    else:
        status = rng.choices(["ACTIVE", "INACTIVE", "SUSPENDED"], weights=[85, 12, 3])[0]
        rating = round(rng.uniform(3.0, 5.0), 2)
        joined = date(2022, 1, 1) + timedelta(days=rng.randint(0, 729))
        kyc = rng.random() < 0.9
    return (
        partner_id, faker.name(), faker.phone_number(), faker.email(),
        rng.choice(CITIES), rng.choice(VEHICLE_TYPES), status, rng.choice(SHIFTS),
        rating, joined, kyc,
    )


def seed_partners_once():
    """Add the first 5,001 partners — only when the table is still empty."""
    with get_lakebase_connection() as conn:
        existing = conn.execute("SELECT count(*) FROM zomato_ops.delivery_partners").fetchone()[0]
        if existing > 0:
            print(f"  ✓ Partners already loaded ({existing:,} rows) — seed skipped")
            return
        rng = random.Random(2025)  # fixed seed → the same partners every time
        faker = Faker("en_IN")
        faker.seed_instance(2025)
        rows = [make_partner(i, rng, faker) for i in range(NUM_DELIVERY_PARTNERS)]
        with conn.transaction():   # one transaction for the whole seed
            conn.cursor().executemany(INSERT_PARTNER_SQL, rows)
    print(f"  ✓ Added {len(rows):,} delivery partners")


def pick_random_update(rng, faker):
    """Pick one realistic change. Returns (column_name, new_value)."""
    change = rng.choice(["status", "vehicle", "city", "rating", "shift", "kyc", "phone"])
    if change == "status":
        return "partner_status", rng.choice(["ACTIVE", "INACTIVE", "SUSPENDED"])
    if change == "vehicle":
        return "vehicle_type", rng.choice(VEHICLE_TYPES + ["Electric Scooter"])
    if change == "city":
        return "city", rng.choice(CITIES)
    if change == "rating":
        return "rating", round(rng.uniform(2.5, 5.0), 2)
    if change == "shift":
        return "shift_preference", rng.choice(SHIFTS)
    if change == "kyc":
        return "kyc_verified", True
    return "phone", faker.phone_number()


def simulate_app_changes(num_changes):
    """Behave like the partner app: new partners join, partners change, some leave. Every change sets updated_at."""
    rng = random.Random()  # no seed → different changes on every run
    faker = Faker("en_IN")
    counts = {"new partner": 0, "update": 0, "offboarded": 0}

    with get_lakebase_connection() as conn:
        partner_ids = [row[0] for row in conn.execute(
            "SELECT partner_id FROM zomato_ops.delivery_partners WHERE partner_status <> 'OFFBOARDED'")]
        next_index = conn.execute("SELECT count(*) FROM zomato_ops.delivery_partners").fetchone()[0]

        for _ in range(num_changes):
            action = rng.choices(["new partner", "update", "offboarded"], weights=[15, 80, 5])[0]
            if action == "new partner":
                new_partner = make_partner(next_index, rng, faker, is_new_joiner=True)
                conn.execute(INSERT_PARTNER_SQL, new_partner)
                partner_ids.append(new_partner[0])
                next_index += 1
            elif action == "offboarded":
                partner_id = partner_ids.pop(rng.randrange(len(partner_ids)))
                conn.execute(OFFBOARD_PARTNER_SQL, [partner_id])
            else:
                column, new_value = pick_random_update(rng, faker)
                # column always comes from pick_random_update (our own fixed list), never from user input
                conn.execute(
                    f"UPDATE zomato_ops.delivery_partners SET {column} = %s, updated_at = now() WHERE partner_id = %s",
                    [new_value, rng.choice(partner_ids)],
                )
            counts[action] += 1
    return counts


seed_partners_once()
if NUM_PARTNER_CHANGES > 0:
    print(f"  ✓ App changes this run: {simulate_app_changes(NUM_PARTNER_CHANGES)}")

with get_lakebase_connection() as conn:
    by_status = conn.execute(
        "SELECT partner_status, count(*) FROM zomato_ops.delivery_partners GROUP BY 1 ORDER BY 1").fetchall()
    newest_change = conn.execute("SELECT max(updated_at) FROM zomato_ops.delivery_partners").fetchone()[0]
print(f"\n  Partners in Lakebase by status : {dict(by_status)}")
print(f"  Newest updated_at             : {newest_change}")
print("\n🟢 Lakebase source ready — Bronze will pick up the changes.")